# EXP-028 v2 — сравнение вариантов критика PPO

Сравниваю три варианта критика на 640 ответах разогрева сида 0. Из сохранённых буферов беру оценки закрытых слайдов и скрытые состояния T-lite. Для всех вариантов использую одни и те же позиции токенов, итоговые награды и отложенные темы.

Для префиксных признаков использую историю до оцениваемого токена. Слайд входит в оценку после появления следующей завершённой строки `#SLIDE`.

## 1. Настройки и данные

Читаю `episodes.jsonl` из разогрева сида 0. Проверяю, что доступны все ответы. Кеш префиксных признаков привязываю к данным, ревизии токенизатора и коду их расчёта.

In [ ]:
import statistics
import sys
import time
from dataclasses import replace
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from presentation_rl.tasks import load_tasks
from presentation_rl.training.common import RewardModel, file_hash, load_policy, setup_torch, stable_seed, write_json
from presentation_rl.training.ppo import PPOConfig, exp028_verdict, load_warmup_episodes, split_topics, warmup_features
from presentation_rl.training.prefix_features import (
    FEATURE_NAMES, PrefixFeatureExtractor, aligned_reference_features, fit_prefix_head,
    prefix_cache_matches, prefix_warmup_code_hash, prepare_prefix_warmup,
)

SEED = 0
WARMUP = ROOT / "runs" / "ppo" / f"warmup_seed{SEED}"
EPISODES = WARMUP / "episodes.jsonl"
FEATURES = WARMUP / "prefix_features.pt"
REPORT = WARMUP / "exp028_v2_heads.json"
TASKS = ROOT / "data" / "tasks" / "tasks.json"
POINTS = ROOT / "data" / "tasks" / "key_points" / "key_points.json"

config = PPOConfig.load(ROOT / "configs" / "algorithms" / "ppo.yaml", seed=SEED, warmup_dir=str(WARMUP))
episodes = load_warmup_episodes(WARMUP)
per_step = 2 * config.tasks_per_mode * config.group_size
expected = config.warmup_steps * per_step
if len(episodes) != expected:
    raise ValueError(f"В {EPISODES} {len(episodes)} эпизодов, ожидаю {expected}")
tasks = {task.id: task for task in load_tasks(TASKS)}

SRC = ROOT / "src" / "presentation_rl"
score_files = ("dsl.py", "environment.py", "perturb.py", "rewards.py", "key_points.py",
               "rendering/builder.py", "training/common.py", "training/ppo.py")
provenance = {
    "episodes": file_hash(EPISODES), "tasks": file_hash(TASKS), "points": file_hash(POINTS),
    "code": {name: file_hash(SRC / name) for name in score_files}
            | {"training/prefix_features.py": prefix_warmup_code_hash()},
    "model": config.policy_model, "revision": config.policy_revision,
    "max_new_tokens": config.max_new_tokens, "max_context": config.max_context,
    "reward_floor": config.reward_floor, "max_positions": config.warmup_max_positions,
    "embedder_device": config.embedder_device, "seed": SEED, "feature_names": list(FEATURE_NAMES),
}
print(f"Разогрев: {len(episodes)} эпизодов, {config.warmup_steps} пакетов по {per_step}.")

## 2. Префиксные признаки и время расчёта

Читаю готовый `prefix_features.pt`, если входные данные и извлечение признаков совпадают. Показываю время исходного расчёта и оценку расходов на два запуска PPO. Если буфер нужно пересчитать, использую bge-m3 на GPU 1 и собираю закрытые слайды в памяти.

In [ ]:
cached = torch.load(FEATURES, map_location="cpu", weights_only=True) if FEATURES.exists() else None
if cached and prefix_cache_matches(cached.get("provenance", {}), provenance):
    data = cached["data"]
    print(f"Читаю готовые признаки из {FEATURES.name}. Ниже время их исходного расчёта.")
else:
    from transformers import AutoTokenizer

    began = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(config.policy_model, revision=config.policy_revision)
    reward = RewardModel(ROOT, device=config.embedder_device, floor=config.reward_floor)
    extract = PrefixFeatureExtractor(
        tokenizer, reward, max_new_tokens=config.max_new_tokens, max_context=config.max_context,
    )
    print(f"Загрузка токенизатора и bge-m3: {time.perf_counter() - began:.1f} с")

    began = time.perf_counter()
    data = prepare_prefix_warmup(episodes, tasks, extract, config)
    tmp = FEATURES.with_suffix(".pt.tmp")
    torch.save({"provenance": provenance, "data": data}, tmp)
    tmp.replace(FEATURES)
    print(f"Расчёт и сохранение: {(time.perf_counter() - began) / 60:.1f} мин · {FEATURES.name}")
    del extract, reward, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

times = [row["seconds"] for row in data["timing"]]
step_seconds = statistics.fmean(times)
step_scores = statistics.fmean(row["score_calls"] for row in data["timing"])
print(f"Буфер: {len(data['features'])} эпизодов · {sum(map(len, data['features']))} позиций · "
      f"{len(data['feature_names'])} признаков")
print(f"Извлечение на разогреве: {sum(times) / 60:.1f} мин, самый долгий пакет {max(times):.1f} с")
print(f"Добавка на шаг PPO (оценка): {step_seconds:.1f} с · около {step_scores:.0f} оценок")
print(f"Добавка на два сида по {config.steps} шагов: {2 * config.steps * step_seconds / 3600:.2f} ч")

## 3. Признаки T-lite

Читаю `features.pt` из первого EXP-028 и сверяю токены, позиции действий, режимы и награды с префиксным буфером. Если подходящего файла нет, считаю скрытые состояния на GPU 0 и сохраняю отдельный буфер для этого эксперимента.

In [ ]:
rebuilt = WARMUP / "exp028_v2_reference"
for reference_file in (WARMUP / "features.pt", rebuilt / "features.pt"):
    if not reference_file.exists():
        continue
    reference_data = torch.load(reference_file, map_location="cpu", weights_only=True)
    try:
        reference = aligned_reference_features(data, reference_data, episodes, config.policy_revision)
    except ValueError as error:
        print(f"{reference_file}: {error}")
        continue
    print(f"Читаю готовые признаки T-lite: {reference_file}")
    break
else:
    print("Считаю признаки T-lite на GPU 0.", flush=True)
    began = time.perf_counter()
    setup_torch()
    rebuilt.mkdir(exist_ok=True)
    model, _ = load_policy(config, ROOT)
    reference_data = warmup_features(model, episodes, config, rebuilt)
    reference = aligned_reference_features(data, reference_data, episodes, config.policy_revision)
    reference_file = rebuilt / "features.pt"
    del model
    torch.cuda.empty_cache()
    print(f"T-lite: {(time.perf_counter() - began) / 60:.1f} мин · {reference_file}")
del reference_data
print(f"Признаки T-lite: {len(reference)} эпизодов · {reference[0].shape[1]} компонент")

## 4. Сравнение вариантов критика

На одних отложенных темах сравниваю три варианта:

- Линейный критик на префиксных признаках.
- Критик с MLP на префиксных признаках (64 скрытые единицы).
- Критик с MLP на префиксных признаках и скрытых состояниях T-lite (256 скрытых единиц).

Префиксные признаки стандартизирую по обучающей части. Первые два варианта обучаю на CPU, третий — на GPU 0.

Эпоху выбираю по MSE, как в исходном EXP-028. Для каждого варианта сравниваю MSE на всей выборке, в обоих режимах и на первом действии с общей константой и средним по режиму. Сохраняю обе парные меры с бутстрепом по темам, вердикт исходного правила и время обучения. Темы уже использовались в первом EXP-028 и при выборе эпохи, поэтому читаю интервалы как результат на наборе разработки.

In [ ]:
holdout = split_topics(data["tasks"], config.warmup_holdout, stable_seed(SEED, "warmup-split"))
print(f"Отложено тем: {len(holdout)} из {len(set(data['tasks']))}")
candidates = (
    ("Линейный критик на префиксных признаках", "linear", None, 1e-3, False),
    ("Критик с MLP на префиксных признаках", "mlp", 64, 1e-3, False),
    ("Критик с MLP на префиксных признаках и скрытых состояниях T-lite", "mlp", 256, 1e-4, True),
)
results = {}
for label, kind, width, lr, use_reference in candidates:
    device = config.head_device if use_reference else "cpu"
    head_config = replace(config, head_device=device, head_hidden=width or config.head_hidden)
    print(f"\n{label} · {device}", flush=True)
    began = time.perf_counter()
    fit = fit_prefix_head(data, head_config, holdout, lr, kind=kind,
                          reference=reference if use_reference else None)
    seconds = time.perf_counter() - began
    metrics = fit["holdout"]
    verdict = exp028_verdict(metrics)

    print(f"Вход {fit['input_dim']} · эпоха {fit['best_epoch']} · обучение и оценка {seconds:.1f} с")
    for baseline, metric in (("Общая константа", "paired"), ("Среднее по режиму", "paired_mode")):
        p = metrics[metric]
        print(f"{baseline} · парная мера {p['estimate']:+.5f} [{p['low']:+.5f}; {p['high']:+.5f}]")
    for scope in ("all", "first", "plain_all", "search_all"):
        if scope + "_mse" in metrics:
            print(f"{scope} · MSE {metrics[scope + '_mse']:.5f} | "
                  f"константа {metrics[scope + '_mse_const']:.5f} | режим {metrics[scope + '_mse_mode']:.5f}")
    print(f"Смещение V − R: {metrics['all_bias']:+.5f} · вердикт: {verdict}")
    results[label] = {
        "hidden": width, "device": device, "uses_prefix": True, "uses_reference": use_reference,
        "fit_seconds": seconds, "verdict": verdict,
        "fit": {name: value for name, value in fit.items() if name not in ("head", "optimizer")},
    }
    del fit

write_json(REPORT, {
    "seed": SEED, "holdout_topics": sorted(holdout), "provenance": provenance,
    "tables": {"prefix": str(FEATURES), "reference": str(reference_file)},
    "timing": data["timing"], "candidates": results,
})
print(REPORT)

## Зафиксированный результат первой проверки линейного критика

Сохраняю вывод первого прогона. Его исходный файл — `exp028_v2.json`. Сравнение трёх вариантов критика записываю в `exp028_v2_heads.json`.

```text
Загрузка токенизатора и bge-m3: 12.9 с
признаки: шаг 1/10 · 89.1 с · 530 оценок
признаки: шаг 2/10 · 54.7 с · 442 оценок
признаки: шаг 3/10 · 199.2 с · 607 оценок
признаки: шаг 4/10 · 149.5 с · 723 оценок
признаки: шаг 5/10 · 43.2 с · 495 оценок
признаки: шаг 6/10 · 51.7 с · 541 оценок
признаки: шаг 7/10 · 41.0 с · 486 оценок
признаки: шаг 8/10 · 49.8 с · 439 оценок
признаки: шаг 9/10 · 95.4 с · 622 оценок
признаки: шаг 10/10 · 36.5 с · 439 оценок
Расчёт и сохранение: 13.5 мин · prefix_features.pt
Буфер: 640 эпизодов · 201195 позиций · 35 признаков
Извлечение на разогреве: 13.5 мин, самый долгий пакет 199.2 с
Добавка на шаг PPO (оценка): 81.0 с · около 532 оценок
Добавка на два сида по 150 шагов: 6.75 ч

Отложено тем: 15 из 73 · лучшая эпоха 4 · обучение и оценка 8.6 с
Общая константа · парная мера +0.01576 [+0.00625; +0.02598]
Среднее по режиму · парная мера +0.01090 [+0.00457; +0.01770]
all · MSE 0.01469 | константа 0.02687 | режим 0.02495
plain_all · MSE 0.00507 | константа 0.01089 | режим 0.01033
search_all · MSE 0.02813 | константа 0.04921 | режим 0.04540
Смещение V − R: +0.01441
Вердикт по исходному правилу EXP-028: данных недостаточно (хуже константы: first)
/home/jovyan/Diplom/runs/ppo/warmup_seed0/exp028_v2.json
```